EAA: Анализ текучести сотрудников с использованием RandomForestClassifier  
Цель: предсказать, кто может уйти из компании и какие признаки на это влияют

In [1]:
import pandas as pd                 
import numpy as np                  
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

Данные
- Файл: `Employee.csv`
- Колонки:
  - `Education` — уровень образования сотрудника
  - `JoiningYear` — год, когда сотрудник присоединился к компании
  - `City` — город, где работает сотрудник
  - `PaymentTier` — уровень оплаты/зарплаты
  - `Age` — возраст сотрудника
  - `Gender` — пол сотрудника
  - `EverBenched` — был ли сотрудник когда-либо на бенче
  - `ExperienceInCurrentDomain` — опыт работы в текущей сфере
  - `LeaveOrNot` — таргет, 1 = ушёл, 0 = остался

In [2]:
# Загружаем CSV с данными
df = pd.read_csv("../data/Employee.csv")  
df.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Education                  4653 non-null   object
 1   JoiningYear                4653 non-null   int64 
 2   City                       4653 non-null   object
 3   PaymentTier                4653 non-null   int64 
 4   Age                        4653 non-null   int64 
 5   Gender                     4653 non-null   object
 6   EverBenched                4653 non-null   object
 7   ExperienceInCurrentDomain  4653 non-null   int64 
 8   LeaveOrNot                 4653 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 327.3+ KB


Education                    0
JoiningYear                  0
City                         0
PaymentTier                  0
Age                          0
Gender                       0
EverBenched                  0
ExperienceInCurrentDomain    0
LeaveOrNot                   0
dtype: int64

In [4]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include='object').columns
print("Категориальные колонки:", categorical_cols)

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

Категориальные колонки: Index(['Education', 'City', 'Gender', 'EverBenched'], dtype='object')


In [5]:
# Таргет
target = 'LeaveOrNot'  
X = df.drop(columns=[target])
y = df[target]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [8]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.849624060150376
Recall: 0.7227414330218068
F1-score: 0.7682119205298014

Confusion Matrix:
 [[559  51]
 [ 89 232]]

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.92      0.89       610
           1       0.82      0.72      0.77       321

    accuracy                           0.85       931
   macro avg       0.84      0.82      0.83       931
weighted avg       0.85      0.85      0.85       931



In [9]:
results = X_test.copy()
results['actual'] = y_test
results['predicted'] = y_pred
results[results['predicted'] == 1]  # 1 = уходящие

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,actual,predicted
1417,0,2016,2,3,27,0,0,5,1,1
1034,0,2015,2,2,25,0,1,3,1,1
1094,0,2015,2,2,27,0,0,5,1,1
4036,0,2018,2,2,39,0,0,4,1,1
1128,0,2015,2,2,26,0,0,4,1,1
...,...,...,...,...,...,...,...,...,...,...
3800,0,2017,2,3,34,0,0,1,0,1
4434,1,2013,1,2,35,0,0,2,1,1
2447,0,2015,2,3,27,0,0,5,0,1
4572,0,2014,2,1,22,0,0,0,1,1
